# ネプリライシン阻害剤複合体 (1R1H/1R1I/1R1J) での chem.protein.split() デモ

同じ標的タンパク質(ネプリライシン, neutral endopeptidase/NEP)に異なる低分子阻害剤が結合した
RCSBの3構造 `1R1H`/`1R1I`/`1R1J` を題材に、新しく実装した `chem.protein.split()` の動作を確認する。
3構造とも単一チェーンA、糖鎖修飾 `NAG` ×3・亜鉛イオン `ZN` ×1・低分子阻害剤(それぞれ
`BIR`/`TI1`/`OIR`)という同じ構成のヘテロ原子グループを持ち、「リガンドを含まないPDB」と
「水以外の各HETATM残基インスタンスを結合次数まで復元したSDF」への分割を、複数構造にわたって
横断的に確認するのに適している。


In [ ]:
from chem import rcsb

entry_ids = ["1R1H", "1R1I", "1R1J"]
rcsb.download_structures(entry_ids, outdir="neprilysin_data", filetype="pdb")


## chem.protein.split() で構造を分割する

`chem.protein.split()` は構造ファイルを以下の2つに分割する:

1. **リガンドフリーの蛋白質PDB** -- 結晶水は残したまま、それ以外のHETATM残基
   (低分子リガンドはもちろん、亜鉛イオンや糖鎖修飾も)を全て取り除いたもの
   (`split_chains=True` にすると、蛋白質PDBもチェーンごとに、ファイル名にチェーンIDを
   含めて分割される)
2. **リガンドSDF** -- 水を除く各HETATM残基インスタンスごとに1ファイル。3D座標は元の構造
   そのまま、結合次数・芳香族性はPDB Chemical Component Dictionaryのテンプレートと照合して
   復元される(`chem.ligand.load_ligand` と同じロジック)

`chem.ligand.load_ligand` と同様、テンプレートと原子数が一致しないインスタンス(共有結合した
ペプチド様リガンドや、密度が不完全な残基など)は例外にはせず、警告付きでスキップされる。


In [ ]:
import os

from chem import protein

split_results = {}
for entry_id in entry_ids:
    split_results[entry_id] = protein.split(
        os.path.join("neprilysin_data", f"{entry_id}.pdb"),
        outdir="neprilysin_split",
    )
split_results["1R1H"]


### 分割結果の一覧

`chem.ligand.list_ligand_instances` (`exclude=chem.protein.WATER`) で各構造の水以外の
HETATM残基インスタエンスを全て列挙し、それぞれが `split()` によって実際にSDF化された
(`sdf_written`) かどうかを突き合わせる。3構造とも `NAG`(N-結合型糖鎖、3残基)はRDKitの
テンプレートマッチングの既知の制限でスキップされ、`ZN` と主要阻害剤(`BIR`/`TI1`/`OIR`)は
正しくSDF化されることを確認する。


In [ ]:
import pandas as pd

from chem.ligand import list_ligand_instances
from chem.protein import WATER

rows = []
for entry_id in entry_ids:
    all_instances = list_ligand_instances(
        os.path.join("neprilysin_data", f"{entry_id}.pdb"), exclude=WATER
    )
    written = {(l["code"], l["chain"], l["resnum"]) for l in split_results[entry_id]["ligands"]}
    for inst in all_instances:
        key = (inst["code"], inst["chain"], inst["resnum"])
        rows.append({"entry_id": entry_id, **inst, "sdf_written": key in written})

ligands_df = pd.DataFrame(rows)
ligands_df


### リガンドフリー蛋白質PDBの中身を確認

分割後の蛋白質PDBに対して改めて `list_ligand_instances` を実行し、水以外のHETATM残基が
本当に1つも残っていないことを確認する。


In [ ]:
# NAG (スキップされた) も含め、水以外のHETATM残基が1つも残っていないことを確認
for entry_id in entry_ids:
    remaining = list_ligand_instances(split_results[entry_id]["protein"], exclude=WATER)
    print(entry_id, "remaining non-water HETATM residues:", remaining)


### 3種類の阻害剤SDFを比較する(分子量・QED・芳香族性)

主要阻害剤(`BIR`/`TI1`/`OIR`)のSDFを読み込み直し、芳香環が `GetIsAromatic()` で
芳香族として認識される(=結合次数が正しく復元されている)ことと、`chem.ligand` の
物性計算関数(`molecular_weight`/`qed`)をまとめて確認する。


In [ ]:
from rdkit import Chem

from chem import ligand

main_ligand_code = {"1R1H": "BIR", "1R1I": "TI1", "1R1J": "OIR"}

rows = []
for entry_id, code in main_ligand_code.items():
    lig_entry = next(l for l in split_results[entry_id]["ligands"] if l["code"] == code)
    mol = next(Chem.SDMolSupplier(lig_entry["path"]))
    rows.append(
        {
            "entry_id": entry_id,
            "code": code,
            "n_atoms": mol.GetNumAtoms(),
            "n_aromatic_atoms": sum(atom.GetIsAromatic() for atom in mol.GetAtoms()),
            "molecular_weight": round(ligand.molecular_weight(mol), 1),
            "qed": round(ligand.qed(mol), 3),
            "smiles": Chem.MolToSmiles(mol),
        }
    )
pd.DataFrame(rows)


### 可視化: リガンドフリー蛋白質 + 分割後のSDFを重ねて表示

`split()` はリガンドの3D座標を元の構造からそのままコピーするだけなので、分割後の蛋白質PDBと
リガンドSDFを同じ py3Dmol ビューに重ねれば、座標系がずれずに元の複合体そのままの配置で
表示できるはずである。ここでは `1R1H` のリガンドフリー蛋白質(cartoon)と `BIR` 阻害剤の
SDF(stick)を重ねてそれを確認する。


In [ ]:
import py3Dmol

protein_path = split_results["1R1H"]["protein"]
bir_path = next(l["path"] for l in split_results["1R1H"]["ligands"] if l["code"] == "BIR")

with open(protein_path) as f:
    protein_pdb_text = f.read()
with open(bir_path) as f:
    bir_sdf_text = f.read()

view = py3Dmol.view(width=600, height=450)
view.addModel(protein_pdb_text, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})
view.addModel(bir_sdf_text, "sdf")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo({"model": 1})
view.show()
